# Import Packages

In [1]:
import os
import re
import logging
import numpy as np
import pandas as pd
from io import StringIO
from sklearn.metrics import confusion_matrix, classification_report
from IPython.display import clear_output
import matplotlib.pyplot as plt
from bci_essentials.signal_processing import bandpass
from scipy.signal import welch


from bci_essentials.io.xdf_sources import XdfEegSource, XdfMarkerSource
from bci_essentials.bci_controller import BciController
from bci_essentials.paradigm.ssvep_paradigm import SsvepParadigm
from bci_essentials.data_tank.data_tank import DataTank
from bci_essentials.utils.logger import Logger  # Logger wrapper
from bci_essentials.classification.ssvep_fbcca_classifier import (
    SsvepFbCcaClassifier,
)

%matplotlib qt

# Import Data

In [2]:
filename = os.path.join("data", "sub-P004_ses-S001_task-T2_run-001_eeg.xdf")

logger = Logger(name=__name__)
logger.setLevel(1)
eeg_source = XdfEegSource(filename)
marker_source = XdfMarkerSource(filename)
paradigm = SsvepParadigm(live_update=False, iterative_training=False, filters = [1,45]) # Setting live_upadate to False means it will classify each trial rather than each epoch (full 5 seconds rather than every second 5 time)
data_tank = DataTank()


2025-06-10 13:12:37 - INFO - __main__ : Setting logging level to Unknown Level


# Define Classifier

In [3]:
# Settings
target_frequencies = np.array([6.25, 10, 11.11111, 14.28571])

n_harmonics = 3  # Number of harmonics to use for SSVEP classification
fb_cutoffs = np.array([[i, 27] for i in range(3, 25, 2)])   # Filter bank cut-off frequencies 43 and 45
#filter_order = 6
nsamples = 5 * 256  # 5 seconds of data at 256 Hz
fsample = 256.0  # Hz
concatenate_trials = True  # Concatenate trials for training

classifier = SsvepFbCcaClassifier(subset=["O1", "Oz", "O2"])
classifier.set_ssvep_settings(
    fsample=fsample,
    n_harmonics=n_harmonics,
    target_freqs=target_frequencies,
    n_samples=nsamples,
    filter_bank=fb_cutoffs,
    concatenate_trials=concatenate_trials,
)

test_ssvep = BciController(classifier, eeg_source, marker_source, paradigm, data_tank)

test_ssvep.setup(
    online=False,
    train_complete=True,  # Set to True to run the classifier on the entire dataset
)

2025-06-10 13:12:38 - INFO - bci_essentials.classification.generic_classifier : Initializing the classifier
2025-06-10 13:12:38 - INFO - bci_essentials.bci_controller : g.USBamp-1
2025-06-10 13:12:38 - INFO - bci_essentials.bci_controller : ['Fz', 'F4', 'F8', 'C3', 'Cz', 'C4', 'T8', 'P7', 'P3', 'P4', 'P8', 'PO7', 'PO8', 'O1', 'Oz', 'O2']


# Run Classifier

In [4]:
# Create a string buffer to capture log output
log_stream = StringIO()
handler = logging.StreamHandler(log_stream)
logger = logging.getLogger()
logger.addHandler(handler)

# Run the test
test_ssvep.run()

# Get log output as string
log_output = log_stream.getvalue()

# Extract true and predicted labels using regex
y_true = []
y_pred = []

for line in log_output.split('\n'):
    # Extract true labels
    if "Cued index: " in line:
        true_label = int(re.search(r"Cued index: (\d+)", line).group(1))
        y_true.append(true_label)
    
    # Extract predicted labels
    if "Predicted label: " in line:
        pred_label = int(re.search(r"Predicted label: (\d+)", line).group(1))
        y_pred.append(pred_label)

clear_output()

# Convert lists to numpy arrays
y_true = np.array(y_true) #y_true = np.array(y_true[:-1])
y_pred = np.array(y_pred)

# Calculate and display accuracy
accuracy = np.mean(y_true == y_pred)
print(f"Accuracy: {accuracy:.2%}")

# Create confusion matrix
conf_matrix = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix:")
print(conf_matrix)

# Print detailed classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred))
print(f"Number of samples: {len(y_true)}")

# Get the target frequencies, note that this must be done after run, so that the classifier data fields are populated
target_freqs = classifier.target_freqs

# Log the target freqs
logger.info("The target frequencies are: %s", target_freqs)

Accuracy: 100.00%

Confusion Matrix:
[[9 0 0 0]
 [0 7 0 0]
 [0 0 8 0]
 [0 0 0 6]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         9
           1       1.00      1.00      1.00         7
           2       1.00      1.00      1.00         8
           3       1.00      1.00      1.00         6

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30

Number of samples: 30


# Apply Hanning window to EEG and reshape into 30 5 second epochs
- originally 150 1 second epochs

In [5]:
epochs = data_tank.get_epochs()
eeg_epochs = epochs[0] # 0 index contains the EEG data

# Get dimensions
[_, n_channels, n_samples] = eeg_epochs.shape
    
# Create and apply Hanning window to all trials at once
window = np.hanning(n_samples)
windowed_data = eeg_epochs * window[np.newaxis, np.newaxis, :]  

# Reshape the data
temp_reshape = windowed_data.reshape(30, 5, 16, 256) 
temp_reshape = np.transpose(temp_reshape, (0, 2, 1, 3)) 
eeg_reshape = temp_reshape.reshape(30, 16, -1)

print("Final shape:", np.shape(eeg_reshape)) 

Final shape: (30, 16, 1280)


# Sort Markers

In [6]:
def process_marker(row):
    if "Cued index" in row[0]:
        return int(row[0].split(":")[-1].strip())
    return None

def update_ssvep_marker(row, cued_index):
    if "ssvep" in row[0]:
        parts = row[0].split(",")
        parts[2] = str(cued_index)
        return ",".join(parts)
    return row[0]

cued_index = None
updated_rows = []

markers = pd.DataFrame(log_output.split('\n'))

for _, row in markers.iterrows():
    new_cued_index = process_marker(row)
    if new_cued_index is not None:
        cued_index = new_cued_index
    updated_rows.append([update_ssvep_marker(row, cued_index)])

markers = pd.DataFrame(updated_rows, columns=["Answer"]).query("Answer.str.contains('ssvep')", engine="python") # Retain only rows that contain 'ssvep'
markers = markers.iloc[::5].reset_index(drop=True)  # Retain one of every 5 rows

# Combine EEG epochs and markers

In [7]:
trial_data = list(zip(eeg_reshape, markers["Answer"].values))

# Plot PSD for O channels for each target frequency
## Individual plots

In [8]:
channel_indices = {'O1': 13, 'Oz': 14, 'O2': 15} # 'Oz': 14, 

trials = []
labels = []

for eeg_group, marker_df in trial_data:
    trials.append(eeg_group)

    marker_string = marker_df
    parts = marker_string.split(",")
    group_label = int(parts[2])
    labels.append(group_label)

trials = np.array(trials)  # shape (n_trials, 16, 256)
labels = np.array(labels)

# Plot by unique group label
unique_labels = {0: "6.25 Hz", 1: "10 Hz", 2: "11.11 Hz", 3: "14.28 Hz"}
target_frequencies = {0: 6.25, 1: 10, 2: 11.11, 3: 14.28571} 

for label in unique_labels.keys():
    label_trials = trials[labels == label]  # shape (n, 1280)

    plt.figure(figsize=(10, 6))

    for ch_name, ch_idx in channel_indices.items():
        # Concatenate all data from this channel across trials
        ch_data = label_trials[:, ch_idx, :].reshape(-1) 
        freqs, psd = welch(ch_data, fs=fsample, nperseg = 256*5)

        plt.plot(freqs, psd, label=f'{ch_name} (Avg)', linewidth=2)

    # Target frequency and harmonics
    tf = target_frequencies[label]
    for i in range(1, 4):
        plt.axvline(tf * i, color='b', linestyle='--', linewidth=1)

    plt.title(f'Average PSD - {unique_labels[label]} Trials')
    plt.xlabel('Frequency (Hz)')
    plt.ylabel('Power (µV²/Hz)')
    plt.xlim([4, 45])
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

## Panel plot

In [9]:
trials = []
labels = []

for eeg_group, marker_df in trial_data:
    trials.append(eeg_group)  # shape (16, 1280)

    marker_string = marker_df
    parts = marker_string.split(",")
    group_label = int(parts[2])
    labels.append(group_label)

trials = np.array(trials)  # shape (30, 16, 1280) - preserves (trials, channels, samples)
labels = np.array(labels)

# Create panel plot
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)
axes = axes.flatten()

for idx, label in enumerate(unique_labels.keys()):
    ax = axes[idx]
    label_trials = trials[labels == label]  # shape (n, 16, 1280)

    for ch_name, ch_idx in channel_indices.items():
        # Get data for specific channel across all trials with this label
        ch_data = label_trials[:, ch_idx, :].reshape(-1)
        freqs, psd = welch(ch_data, fs=fsample, nperseg=5*256)  # Added nperseg parameter
        ax.plot(freqs, psd, label=f'{ch_name}', linewidth=2)

    tf = target_frequencies[label]
    for i in range(1, 4):
        ax.axvline(tf * i, color='b', linestyle='--', linewidth=1)

    ax.set_title(f'{unique_labels[label]}')
    ax.set_xlim([4, 45])
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Power (µV²/Hz)')
    ax.grid(True)
    ax.legend()

plt.suptitle('Average PSD by Frequency and Channel (O1, Oz, O2)', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()